In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family']        = 'serif'
plt.rcParams['font.serif']         = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth']     = 2
plt.rcParams['xtick.major.width']  = 2
plt.rcParams['ytick.major.width']  = 2

# ══════════════════════════════════════════════════════════
# 1. MIC分析
# ══════════════════════════════════════════════════════════
def _mutual_info_grid(x, y, bx, by):
    n   = len(x)
    xy  = np.histogram2d(x, y, bins=[bx, by])[0]
    pxy = xy / n
    px  = pxy.sum(axis=1, keepdims=True)
    py  = pxy.sum(axis=0, keepdims=True)
    mask = pxy > 0
    mi   = np.sum(pxy[mask] * np.log2(pxy[mask] / (px * py + 1e-12)[mask]))
    return max(mi, 0.0)

def compute_mic(x, y, alpha=0.6, c=15):
    n = len(x)
    B = max(int(n ** alpha), 4)
    mic = 0.0
    for bx in range(2, B + 1):
        max_by = max(2, min(B // bx, B))
        for by in range(2, max_by + 1):
            if bx * by > B * c:
                continue
            mi   = _mutual_info_grid(x, y, bx, by)
            norm = np.log2(min(bx, by))
            if norm > 0:
                mic = max(mic, mi / norm)
    return min(mic, 1.0)

def compute_mic_all(X, y):
    n_feat = X.shape[1]
    scores = []
    for i in range(n_feat):
        scores.append(compute_mic(X[:, i], y))
        if (i + 1) % 10 == 0 or (i + 1) == n_feat:
            print(f"  MIC 计算进度: {i+1}/{n_feat}", end='\r')
    print()
    return np.array(scores)

# ══════════════════════════════════════════════════════════
# 2. 加载数据
# ══════════════════════════════════════════════════════════
df = pd.read_excel('B0005_特征数据集.xlsx')          
target_col   = df.columns[-1]
feature_cols = df.columns[1:-1].tolist()
print(f"原始特征数量: {len(feature_cols)}  |  目标变量: {target_col}")

# ══════════════════════════════════════════════════════════
# 3. 训练 / 测试集划分
# ══════════════════════════════════════════════════════════
TRAIN_RATIO = 0.6
split_idx   = int(len(df) * TRAIN_RATIO)
train_df    = df.iloc[:split_idx].copy()
test_df     = df.iloc[split_idx:].copy()
print(f"总循环数: {len(df)}  |  训练集: {len(train_df)}  |  测试集: {len(test_df)}")

# ══════════════════════════════════════════════════════════
# 4. 仅在训练集上 fit 归一化
# ══════════════════════════════════════════════════════════
scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
y_train = train_df[target_col].values

# ══════════════════════════════════════════════════════════
# 5. 计算 MIC
# ══════════════════════════════════════════════════════════
print("\n正在计算 MIC（值域 [0, 1]）...")
mic_scores = compute_mic_all(X_train, y_train)

mic_df = pd.DataFrame({
    'Feature': feature_cols,
    'MIC':     mic_scores
}).sort_values('MIC', ascending=False).reset_index(drop=True)
mic_df['Rank'] = mic_df.index + 1

print("\n── MIC 相关性排名（前10）──")
print(mic_df.head(10).to_string(index=False))

# ══════════════════════════════════════════════════════════
# 6. 可视化
# ══════════════════════════════════════════════════════════
cmap_warm = LinearSegmentedColormap.from_list(
    'warm', ['#5B2D8E', '#C2529A', '#F4845F', '#FBCE6A']
)
def mic_color(v, vmax):
    return cmap_warm(v / vmax)
vmax   = 1.0
colors = [mic_color(v, vmax) for v in mic_df['MIC'].values]

mic_vals  = mic_df['MIC'].values[::-1]
feat_lbls = mic_df['Feature'].values[::-1]

n_feat = len(mic_vals)
fig = plt.figure(figsize=(14, n_feat * 0.35), facecolor='white')
ax1 = fig.add_subplot(1, 1, 1)
fig.subplots_adjust(left=0.12, right=0.95, top=0.98, bottom=0.04)

ax1.set_facecolor('white')
bar_clrs  = colors[::-1]
bars = ax1.barh(range(len(mic_vals)), mic_vals,
                color=bar_clrs, edgecolor='white',
                linewidth=0.4, height=0.9)

for bar, val in zip(bars, mic_vals):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', ha='left',
             fontsize=14, color='#333333')

mic_threshold = 0.75
ax1.axvline(mic_threshold, color='#E53935', lw=1.3, linestyle='--',
            alpha=0.9, label=f'MIC Threshold = {mic_threshold}')
ax1.text(mic_threshold + 0.01, 2, 'Correlation\nThreshold',
         color='#F4845F', fontsize=18, fontstyle='italic', va='bottom')

ax1.set_yticks(range(len(feat_lbls)))
ax1.set_yticklabels(feat_lbls, fontsize=14)
ax1.set_xlabel('MIC Score', fontsize=20, labelpad=8)
ax1.set_xlim(0, 1.08)

for sp in ax1.spines.values():
    sp.set_visible(True)
    sp.set_linewidth(1.2)
ax1.tick_params(axis='x', labelsize=18)

plt.savefig('MIC_analysis.png',
            dpi=300, bbox_inches='tight',
            facecolor='white', transparent=False)
plt.close()
print("\n✅ 图像已保存：MIC_相关性分析.png")

# ══════════════════════════════════════════════════════════
# 7. 导出结果到 Excel
# ══════════════════════════════════════════════════════════
with pd.ExcelWriter('MIC_相关性结果_statistical.xlsx',
                    engine='openpyxl') as writer:

    mic_df[['Rank', 'Feature', 'MIC']]\
        .to_excel(writer, sheet_name='MIC排名', index=False)

    pd.DataFrame({
        '项目': ['总循环数', '训练集循环数', '测试集循环数',
                  '划分比例', '特征数量',
                  'MIC均值', 'MIC最大值', 'MIC最小值'],
        '值':   [len(df), len(train_df), len(test_df),
                  f'{TRAIN_RATIO:.0%}', len(feature_cols),
                  round(mic_df['MIC'].mean(), 4),
                  round(mic_df['MIC'].max(),  4),
                  round(mic_df['MIC'].min(),  4)]
    }).to_excel(writer, sheet_name='实验设置', index=False)

print("✅ 结果已保存：MIC_相关性结果_真实值域.xlsx")